# Contrastive Loss Comparison — Full Experiment Notebook

**Полная сетка:** 3 модели × 3 режима × 3 лосса × 4 LR = **108 запусков**

| | |
|---|---|
| **Модели** | MiniLM-L6, BERT-base, RoBERTa-large |
| **Лоссы** | InfoNCE, Triplet, Cosine |
| **Датасеты** | NLI only, STS only, NLI + STS |
| **LR** | 1e-5, 2e-5, 3e-5, 5e-5 |
| **Эпохи** | 10 (чекпоинт по лучшей валидации) |

**STS-only по всем трём лоссам:**
- `info_nce` — пары с `score ≥ 0.5` как positives, in-batch negatives
- `triplet` — `(anchor, positive)` при `score ≥ 0.6`, `negative` при `score ≤ 0.3`
- `cosine` — MSE(cosine_sim, normalized_human_score)

**Порядок:** [0] Setup → [1] Данные → [2] Baseline → [3] Запуск → [4] Результаты

---
## [0] Setup

In [ ]:
import sys, os

# ── Находим корень проекта по маркеру config.py ──────────────────────────────
def _find_project_root(marker='config.py', max_depth=6):
    """Ищем вверх по дереву каталогов пока не найдём marker-файл."""
    path = os.path.abspath('.')
    for _ in range(max_depth):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(f"Не найден '{marker}' в {max_depth} уровнях выше текущей папки. "
                       f"Текущая папка: {os.path.abspath('.')}")

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Working dir:  {os.getcwd()}')

import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

from config import ExperimentConfig, LossConfig, TrainingConfig, get_run_dir, BACKBONE_SHORT_NAMES
from training.trainer import train_one_run
from training.train_utils import get_device, set_seed

sns.set_theme(style='whitegrid', palette='tab10')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

DEVICE = get_device()
OUTPUT_DIR = 'outputs'
os.makedirs(os.path.join(OUTPUT_DIR, 'plots'), exist_ok=True)

MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'bert-base-uncased',
    'roberta-large',
]
LOSSES = ['info_nce', 'triplet', 'cosine']
MODES  = ['nli_only', 'sts_only', 'nli_plus_sts']
LRS    = [1e-5, 2e-5, 3e-5, 5e-5]
EPOCHS = 10
NUM_TRAINABLE_LAYERS = 2  # 0 = полное fine-tuning; 2 = только последние 2 блока

COLORS_LOSS  = {'info_nce': '#1f77b4', 'triplet': '#ff7f0e', 'cosine': '#2ca02c'}
COLORS_MODEL = {'MiniLM-L6': '#9467bd', 'BERT-base': '#1f77b4', 'RoBERTa-large': '#d62728'}
COLORS_MODE  = {'nli_only': '#1f77b4', 'sts_only': '#ff7f0e', 'nli_plus_sts': '#2ca02c'}
LOSS_LABELS  = {'info_nce': 'InfoNCE', 'triplet': 'Triplet', 'cosine': 'Cosine'}
MODE_LABELS  = {'nli_only': 'NLI only', 'sts_only': 'STS only', 'nli_plus_sts': 'NLI + STS'}

print(f'Device: {DEVICE}')
print('Setup готов.')

In [ ]:
# ── Хелперы ──────────────────────────────────────────────────────────────────
# Значения по умолчанию — продублированы здесь на случай запуска ячейки отдельно
EPOCHS               = globals().get('EPOCHS', 10)
NUM_TRAINABLE_LAYERS = globals().get('NUM_TRAINABLE_LAYERS', 2)  # 0 = полное fine-tuning

def short(model_name):
    return BACKBONE_SHORT_NAMES.get(model_name, model_name.split('/')[-1])


def make_cfg(model_name, loss_type, training_mode, lr, epochs=None, max_train_samples=None):
    _epochs = epochs if epochs is not None else EPOCHS
    return ExperimentConfig(
        model_name=model_name,
        pooling='mean',
        loss=LossConfig(type=loss_type),
        training=TrainingConfig(
            epochs=_epochs,
            sts_finetune_epochs=2,
            batch_size=64 if 'large' not in model_name else 32,
            lr=lr,
            fp16=True,
            warmup_steps=100,
            max_train_samples=max_train_samples,
            num_trainable_layers=NUM_TRAINABLE_LAYERS,
        ),
        training_mode=training_mode,
        output_dir=OUTPUT_DIR,
        seed=42,
    )


def run_one(model_name, loss_type, training_mode, lr,
            epochs=None, max_train_samples=None, skip_if_exists=True):
    cfg = make_cfg(model_name, loss_type, training_mode, lr, epochs, max_train_samples)
    run_dir = get_run_dir(OUTPUT_DIR, loss_type, lr, training_mode, model_name)
    result_path = os.path.join(run_dir, 'metrics', 'result.json')

    if skip_if_exists and os.path.exists(result_path):
        with open(result_path) as f:
            r = json.load(f)
        r['_run_dir'] = run_dir
        r['_skipped'] = True
        return r

    result = train_one_run(cfg, lr=lr, run_dir=run_dir)
    result['_run_dir'] = run_dir
    result['_skipped'] = False
    with open(result_path, 'w') as f:
        json.dump({k: v for k, v in result.items() if not k.startswith('_')}, f, indent=2)
    return result


def load_all_results():
    rows = []
    for root, _, files in os.walk(OUTPUT_DIR):
        if 'result.json' in files:
            try:
                with open(os.path.join(root, 'result.json')) as f:
                    r = json.load(f)
                r['_run_dir'] = root
                r['backbone'] = short(r.get('model_name', ''))
                rows.append(r)
            except Exception:
                pass
    return pd.DataFrame(rows) if rows else pd.DataFrame()


def load_logs(run_dir):
    tp = os.path.join(run_dir, '..', 'logs', 'train_log.csv')
    vp = os.path.join(run_dir, '..', 'logs', 'val_log.csv')
    train_df = pd.read_csv(tp) if os.path.exists(tp) else pd.DataFrame()
    val_df   = pd.read_csv(vp) if os.path.exists(vp) else pd.DataFrame()
    return train_df, val_df


print(f'Хелперы загружены. EPOCHS={EPOCHS}, NUM_TRAINABLE_LAYERS={NUM_TRAINABLE_LAYERS}')

In [ ]:
# ── Хелперы ──────────────────────────────────────────────────────────────────

def short(model_name):
    return BACKBONE_SHORT_NAMES.get(model_name, model_name.split('/')[-1])


def make_cfg(model_name, loss_type, training_mode, lr, epochs=EPOCHS, max_train_samples=None):
    return ExperimentConfig(
        model_name=model_name,
        pooling='mean',
        loss=LossConfig(type=loss_type),
        training=TrainingConfig(
            epochs=epochs,
            sts_finetune_epochs=2,
            batch_size=64 if 'large' not in model_name else 32,
            lr=lr,
            fp16=True,
            warmup_steps=100,
            max_train_samples=max_train_samples,
        ),
        training_mode=training_mode,
        output_dir=OUTPUT_DIR,
        seed=42,
    )


def run_one(model_name, loss_type, training_mode, lr,
            epochs=EPOCHS, max_train_samples=None, skip_if_exists=True):
    cfg = make_cfg(model_name, loss_type, training_mode, lr, epochs, max_train_samples)
    run_dir = get_run_dir(OUTPUT_DIR, loss_type, lr, training_mode, model_name)
    result_path = os.path.join(run_dir, 'metrics', 'result.json')

    if skip_if_exists and os.path.exists(result_path):
        with open(result_path) as f:
            r = json.load(f)
        r['_run_dir'] = run_dir
        return r

    result = train_one_run(cfg, lr=lr, run_dir=run_dir)
    result['_run_dir'] = run_dir
    with open(result_path, 'w') as f:
        json.dump({k: v for k, v in result.items() if not k.startswith('_')}, f, indent=2)
    return result


def load_all_results():
    rows = []
    for root, _, files in os.walk(OUTPUT_DIR):
        if 'result.json' in files:
            try:
                with open(os.path.join(root, 'result.json')) as f:
                    r = json.load(f)
                r['_run_dir'] = root
                r['backbone'] = short(r.get('model_name', ''))
                rows.append(r)
            except Exception:
                pass
    return pd.DataFrame(rows) if rows else pd.DataFrame()


def load_logs(run_dir):
    tp = os.path.join(run_dir, '..', 'logs', 'train_log.csv')
    vp = os.path.join(run_dir, '..', 'logs', 'val_log.csv')
    train_df = pd.read_csv(tp) if os.path.exists(tp) else pd.DataFrame()
    val_df   = pd.read_csv(vp) if os.path.exists(vp) else pd.DataFrame()
    return train_df, val_df


print('Хелперы загружены.')

from data_utils.prepare_nli import prepare_and_save
from data_utils.prepare_sts import prepare_sts_and_save

MAX_TRAIN_SAMPLES = None  # None = полный датасет; поставь 20_000 для быстрого теста

prepare_and_save('data/processed', 'data/cache', max_samples=MAX_TRAIN_SAMPLES)
prepare_sts_and_save('data/processed', 'data/cache')
print('Данные готовы.')

In [ ]:
from data_utils.prepare_nli import prepare_and_save
from data_utils.prepare_sts import prepare_sts_and_save

MAX_TRAIN_SAMPLES = None  # None = полный датасет; поставь 20_000 для быстрого теста

prepare_and_save('data/processed', 'data/cache', max_samples=MAX_TRAIN_SAMPLES)
prepare_sts_and_save('data/processed', 'data/cache')
print('Данные готовы.')

# ── Полная сетка 3 × 3 × 4 × 3 = 108 экспериментов ──────────────────────────
#
# sts_only теперь поддерживает все 3 лосса:
#   - info_nce: пары с score >= 0.5 как positives, in-batch negatives
#   - triplet:  (anchor, positive) score >= 0.6, negative score <= 0.3
#   - cosine:   MSE(cosine_sim, human_score)

SKIP_EXISTING          = True   # False = перезапустить всё
MAX_TRAIN_SAMPLES_GRID = None   # None = полный датасет

MODE_LOSS_PAIRS = [(mode, loss) for mode in MODES for loss in LOSSES]
total = len(MODELS) * len(MODE_LOSS_PAIRS) * len(LRS)
print(f'Всего запусков: {total}  ({len(MODELS)} модели × {len(MODES)} режима × {len(LOSSES)} лосса × {len(LRS)} LR)\n')

done_count = 0
for model in MODELS:
    for mode, loss in MODE_LOSS_PAIRS:
        for lr in LRS:
            run_dir = get_run_dir(OUTPUT_DIR, loss, lr, mode, model)
            exists  = os.path.exists(os.path.join(run_dir, 'metrics', 'result.json'))
            status  = '✓' if exists else '·'
            done_count += exists
            print(f'  {status} {short(model):14s} | {loss:8s} | {mode:13s} | lr={lr:.0e}')

print(f'\nГотово: {done_count}/{total}')

In [ ]:
all_results = []
done, skipped, failed = 0, 0, 0

for model_name in MODELS:
    for mode, loss_type in MODE_LOSS_PAIRS:
        for lr in LRS:
            label = f'{short(model_name)} | {loss_type} | {mode} | lr={lr:.0e}'
            try:
                result = run_one(
                    model_name=model_name,
                    loss_type=loss_type,
                    training_mode=mode,
                    lr=lr,
                    epochs=EPOCHS,
                    max_train_samples=MAX_TRAIN_SAMPLES_GRID,
                    skip_if_exists=SKIP_EXISTING,
                )
                all_results.append(result)
                if result.get('_skipped'):
                    skipped += 1
                else:
                    done += 1
                    sp = result.get('best_spearman_test', 0)
                    print(f'  ✓ {label}  →  Spearman={sp:.4f}')
            except Exception as e:
                failed += 1
                print(f'  ✗ {label}  ERROR: {e}')

print(f'\nГотово: {done} новых, {skipped} пропущено, {failed} ошибок')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(baseline_df['backbone'], baseline_df['spearman_test'],
              color=[COLORS_MODEL.get(b, 'steelblue') for b in baseline_df['backbone']],
              alpha=0.85, width=0.5)
for bar, val in zip(bars, baseline_df['spearman_test']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.4f}',
            ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1)
ax.set_ylabel('Spearman (STS test)')
ax.set_title('Baseline — претренированные модели без fine-tuning')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'baseline.png'), dpi=150)
plt.show()

---
## [3] Эксперименты — полная сетка

**Логика сетки:**
- `nli_only` + `nli_plus_sts`: все 3 лосса
- `sts_only`: только `cosine` (STS train маленький, ~5749 пар; trainer всё равно использует MSE-loss)

Итого: 3 модели × (2×3 + 1×1) режимов-лоссов × 4 LR = **84 запуска**

Установи `SKIP_EXISTING = True` чтобы продолжить с прерванного места.

In [ ]:
SKIP_EXISTING    = True   # False = перезапустить все
MAX_TRAIN_SAMPLES_GRID = None  # ограничить выборку для быстрого теста

# Сетка: (mode, loss) пары
MODE_LOSS_PAIRS = (
    [(mode, loss) for mode in ('nli_only', 'nli_plus_sts') for loss in LOSSES] +
    [('sts_only', 'cosine')]
)

total = len(MODELS) * len(MODE_LOSS_PAIRS) * len(LRS)
print(f'Всего запусков: {total}')
for model in MODELS:
    for mode, loss in MODE_LOSS_PAIRS:
        for lr in LRS:
            run_dir = get_run_dir(OUTPUT_DIR, loss, lr, mode, model)
            exists = os.path.exists(os.path.join(run_dir, 'metrics', 'result.json'))
            status = '✓' if exists else '·'
            print(f'  {status} {short(model):14s} {loss:8s} {mode:13s} lr={lr:.0e}')

In [ ]:
all_results = []
done, skipped, failed = 0, 0, 0

for model_name in MODELS:
    for mode, loss_type in MODE_LOSS_PAIRS:
        for lr in LRS:
            label = f'{short(model_name)} | {loss_type} | {mode} | lr={lr:.0e}'
            try:
                result = run_one(
                    model_name=model_name,
                    loss_type=loss_type,
                    training_mode=mode,
                    lr=lr,
                    epochs=EPOCHS,
                    max_train_samples=MAX_TRAIN_SAMPLES_GRID,
                    skip_if_exists=SKIP_EXISTING,
                )
                all_results.append(result)
                was_skipped = result.get('_skipped', False)
                if was_skipped:
                    skipped += 1
                else:
                    done += 1
                    sp = result.get('best_spearman_test', 0)
                    print(f'  ✓ {label}  →  Spearman={sp:.4f}')
            except Exception as e:
                failed += 1
                print(f'  ✗ {label}  ERROR: {e}')

print(f'\nГотово: {done} новых, {skipped} пропущено, {failed} ошибок')

---
## [4] Результаты и графики

In [ ]:
df = load_all_results()
COL = 'best_spearman_test' if 'best_spearman_test' in df.columns else 'best_spearman'
os.makedirs(os.path.join(OUTPUT_DIR, 'plots'), exist_ok=True)
print(f'Загружено: {len(df)} результатов')
display(
    df[['backbone', 'loss_type', 'training_mode', 'learning_rate', COL, 'training_time', 'best_epoch']]
    .sort_values(COL, ascending=False)
    .reset_index(drop=True)
    .head(20)
)

In [ ]:
# ── Таблица 1: лучший результат по каждой (model, loss, mode) ───────────────
best = (
    df.sort_values(COL, ascending=False)
    .groupby(['backbone', 'loss_type', 'training_mode']).first().reset_index()
)
best['Loss']    = best['loss_type'].map(LOSS_LABELS)
best['Mode']    = best['training_mode'].map(MODE_LABELS)
best['Best LR'] = best['learning_rate'].apply(lambda x: f'{x:.0e}')
best['Spearman']= best[COL].apply(lambda x: f'{x:.4f}')

print('=== Лучший результат по каждой комбинации ===')
display(best[['backbone', 'Loss', 'Mode', 'Best LR', 'Spearman']].sort_values(
    ['backbone', 'Mode', 'Loss']))

In [ ]:
# ── Таблица 2: Baseline vs Best trained ─────────────────────────────────────
if os.path.exists(os.path.join(OUTPUT_DIR, 'baseline.csv')):
    bl = pd.read_csv(os.path.join(OUTPUT_DIR, 'baseline.csv'))
    trained_best = (
        df.groupby('backbone')[COL].max().reset_index()
        .rename(columns={COL: 'best_trained'})
    )
    comp = bl.merge(trained_best, on='backbone')
    comp['Gain'] = (comp['best_trained'] - comp['spearman_test']).apply(lambda x: f'+{x:.4f}' if x >= 0 else f'{x:.4f}')
    comp.columns = ['Backbone', 'Model', 'Baseline', 'Best Trained', 'Gain']
    print('=== Baseline vs Best Trained ===')
    display(comp[['Backbone', 'Baseline', 'Best Trained', 'Gain']])

In [ ]:
# ── График 1: Baseline vs Best Trained ───────────────────────────────────────
if os.path.exists(os.path.join(OUTPUT_DIR, 'baseline.csv')):
    bl = pd.read_csv(os.path.join(OUTPUT_DIR, 'baseline.csv'))
    trained_best = df.groupby('backbone')[COL].max().reset_index()
    comp = bl.merge(trained_best, on='backbone')

    x = np.arange(len(comp))
    fig, ax = plt.subplots(figsize=(9, 5))
    bars1 = ax.bar(x - 0.2, comp['spearman_test'], 0.35, label='Baseline (pretrained)', alpha=0.7, color='#aec7e8')
    bars2 = ax.bar(x + 0.2, comp[COL], 0.35, label='Best Fine-tuned', alpha=0.9,
                   color=[COLORS_MODEL.get(b, 'steelblue') for b in comp['backbone']])
    for bar, val in list(zip(bars1, comp['spearman_test'])) + list(zip(bars2, comp[COL])):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.3f}',
                ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(comp['backbone'])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Spearman (STS test)')
    ax.set_title('Baseline vs Best Fine-tuned')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'baseline_vs_trained.png'), dpi=150)
    plt.show()

In [ ]:
# ── График 2: Heatmap (model × loss) лучший Spearman ─────────────────────────
for mode in df['training_mode'].unique():
    sub = df[df['training_mode'] == mode]
    pivot = sub.pivot_table(index='backbone', columns='loss_type', values=COL, aggfunc='max')
    pivot.columns = [LOSS_LABELS.get(c, c) for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(7, 3))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, linewidths=0.5,
                vmin=0, vmax=1)
    ax.set_title(f'Best Spearman — {MODE_LABELS.get(mode, mode)} | Model × Loss')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', f'heatmap_{mode}.png'), dpi=150)
    plt.show()

In [ ]:
# ── График 3: LR Sensitivity — Spearman vs LR для каждой модели ──────────────
for model_name in MODELS:
    sub = df[(df['model_name'] == model_name) & (df['training_mode'] == 'nli_only')]
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 4))
    for loss_type in LOSSES:
        lsub = sub[sub['loss_type'] == loss_type].sort_values('learning_rate')
        if lsub.empty:
            continue
        ax.plot(lsub['learning_rate'].astype(str), lsub[COL], marker='o',
                label=LOSS_LABELS.get(loss_type, loss_type),
                color=COLORS_LOSS.get(loss_type), linewidth=2)
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Spearman (test)')
    ax.set_title(f'LR Sensitivity — {short(model_name)} | NLI only')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', f'lr_sensitivity_{short(model_name)}.png'), dpi=150)
    plt.show()

In [ ]:
# ── График 4: Supervision strategy comparison ─────────────────────────────────
best_per = (
    df.sort_values(COL, ascending=False)
    .groupby(['backbone', 'training_mode', 'loss_type']).first().reset_index()
)

modes = [m for m in MODES if m in best_per['training_mode'].values]
backbones = best_per['backbone'].unique()

for backbone in backbones:
    sub = best_per[best_per['backbone'] == backbone]
    losses_present = [l for l in LOSSES if l in sub['loss_type'].values]

    x = np.arange(len(losses_present))
    width = 0.25
    fig, ax = plt.subplots(figsize=(9, 5))

    for i, mode in enumerate(modes):
        msub = sub[sub['training_mode'] == mode]
        heights = []
        for lt in losses_present:
            row = msub[msub['loss_type'] == lt]
            heights.append(float(row[COL].values[0]) if not row.empty else 0.0)
        bars = ax.bar(x + i*width, heights, width, label=MODE_LABELS.get(mode, mode),
                      color=COLORS_MODE.get(mode), alpha=0.85)
        for bar, h in zip(bars, heights):
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x + width*(len(modes)-1)/2)
    ax.set_xticklabels([LOSS_LABELS.get(l, l) for l in losses_present])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Best Spearman (test)')
    ax.set_title(f'Supervision Strategy — {backbone}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', f'supervision_{backbone}.png'), dpi=150)
    plt.show()

In [ ]:
# ── График 5: Model size comparison ──────────────────────────────────────────
# Лучший Spearman по каждой (model, loss) паре, лучший mode и LR
model_best = (
    df.sort_values(COL, ascending=False)
    .groupby(['backbone', 'loss_type']).first().reset_index()
)
losses_avail = [l for l in LOSSES if l in model_best['loss_type'].values]
backbones_order = [short(m) for m in MODELS if short(m) in model_best['backbone'].values]

x = np.arange(len(backbones_order))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
for i, lt in enumerate(losses_avail):
    sub = model_best[model_best['loss_type'] == lt]
    heights = [float(sub[sub['backbone'] == b][COL].values[0])
               if not sub[sub['backbone'] == b].empty else 0.0
               for b in backbones_order]
    bars = ax.bar(x + i*width, heights, width, label=LOSS_LABELS.get(lt, lt),
                  color=COLORS_LOSS.get(lt), alpha=0.85)
    for bar, h in zip(bars, heights):
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width*(len(losses_avail)-1)/2)
ax.set_xticklabels(backbones_order)
ax.set_ylim(0, 1)
ax.set_ylabel('Best Spearman (test)')
ax.set_title('Backbone Size Comparison')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'model_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── График 6: Convergence curves — одна модель, все LR, одна картинка на loss ─
TARGET_MODEL = 'bert-base-uncased'  # поменяй на нужную
TARGET_MODE  = 'nli_only'
lr_palette   = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for loss_type in LOSSES:
    runs = df[(df['model_name'] == TARGET_MODEL) &
              (df['training_mode'] == TARGET_MODE) &
              (df['loss_type'] == loss_type)].sort_values('learning_rate')
    if runs.empty:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'{LOSS_LABELS[loss_type]} | {short(TARGET_MODEL)} | {MODE_LABELS[TARGET_MODE]} — сходимость при разных LR', fontsize=12)

    for i, (_, row) in enumerate(runs.iterrows()):
        run_dir = row.get('_run_dir', '')
        train_df, val_df = load_logs(run_dir)
        label = f'lr={row["learning_rate"]:.0e}'
        color = lr_palette[i % 4]

        if not train_df.empty and 'step' in train_df.columns:
            axes[0].plot(train_df['step'], train_df['train_loss'], label=label, color=color, linewidth=1.8)
        if not val_df.empty and 'spearman_score' in val_df.columns:
            vdf = val_df[val_df['stage'] == 'nli'] if 'stage' in val_df.columns else val_df
            axes[1].plot(vdf['epoch'], vdf['spearman_score'], marker='o', label=label, color=color, linewidth=1.8)

    axes[0].set_xlabel('Step');  axes[0].set_ylabel('Train Loss');  axes[0].legend(fontsize=9)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Spearman (val)'); axes[1].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'plots', f'convergence_{loss_type}_{short(TARGET_MODEL)}.png'), dpi=150)
    plt.show()

In [ ]:
# ── График 7: Training Time vs Quality ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for backbone in df['backbone'].unique():
    sub = df[df['backbone'] == backbone]
    ax.scatter(sub['training_time'] / 60, sub[COL],
               label=backbone, color=COLORS_MODEL.get(backbone, 'gray'),
               s=60, alpha=0.7)
ax.set_xlabel('Время обучения (мин)')
ax.set_ylabel('Best Spearman (test)')
ax.set_title('Время обучения vs Качество')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'time_vs_quality.png'), dpi=150)
plt.show()

In [ ]:
# ── График 8: Stability (std Spearman по LR) ──────────────────────────────────
nli_df = df[df['training_mode'] == 'nli_only']
sens = nli_df.groupby(['backbone', 'loss_type'])[COL].agg(
    best='max', std='std', range=lambda x: x.max() - x.min()
).reset_index()
sens['Loss'] = sens['loss_type'].map(LOSS_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pivot_std = sens.pivot_table(index='backbone', columns='loss_type', values='std')
pivot_std.columns = [LOSS_LABELS.get(c, c) for c in pivot_std.columns]
sns.heatmap(pivot_std, annot=True, fmt='.3f', cmap='Oranges', ax=axes[0], linewidths=0.5)
axes[0].set_title('Std Spearman по LR (чем меньше — тем стабильнее)')

pivot_best = sens.pivot_table(index='backbone', columns='loss_type', values='best')
pivot_best.columns = [LOSS_LABELS.get(c, c) for c in pivot_best.columns]
sns.heatmap(pivot_best, annot=True, fmt='.3f', cmap='YlGnBu', ax=axes[1], linewidths=0.5)
axes[1].set_title('Best Spearman по всем LR')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'plots', 'stability.png'), dpi=150)
plt.show()

valid_std = sens['std'].dropna()
if len(valid_std) > 0:
    worst_row = sens.loc[valid_std.idxmax()]
    best_row  = sens.loc[valid_std.idxmin()]
    print(f'Наиболее чувствительна к LR: {worst_row["Loss"]} на {worst_row["backbone"]} (std={worst_row["std"]:.4f})')
    print(f'Наиболее стабильна:          {best_row["Loss"]} на {best_row["backbone"]} (std={best_row["std"]:.4f})')
else:
    print('(запустите все 4 LR для анализа стабильности)')

In [ ]:
# ── Итоговые выводы ────────────────────────────────────────────────────────────
print('=' * 60)
print('ИТОГОВЫЕ ВЫВОДЫ')
print('=' * 60)

# Общий лучший
best_overall = df.loc[df[COL].idxmax()]
print(f'\nЛучший результат:')
print(f'  Модель:   {best_overall["backbone"]}')
print(f'  Loss:     {LOSS_LABELS.get(best_overall["loss_type"], best_overall["loss_type"])}')
print(f'  Режим:    {MODE_LABELS.get(best_overall["training_mode"], best_overall["training_mode"])}')
print(f'  LR:       {best_overall["learning_rate"]:.0e}')
print(f'  Spearman: {best_overall[COL]:.4f}')

# Baseline vs trained
if os.path.exists(os.path.join(OUTPUT_DIR, 'baseline.csv')):
    bl = pd.read_csv(os.path.join(OUTPUT_DIR, 'baseline.csv'))
    print(f'\nПрирост vs baseline:')
    for _, row in bl.iterrows():
        b = df[df['backbone'] == row['backbone']]
        if b.empty: continue
        gain = b[COL].max() - row['spearman_test']
        print(f'  {row["backbone"]:15s}: {row["spearman_test"]:.4f} → {b[COL].max():.4f}  (+{gain:.4f})')

# Best per mode
print(f'\nЛучшая loss function по режимам (NLI only):')
for backbone in df['backbone'].unique():
    sub = df[(df['training_mode'] == 'nli_only') & (df['backbone'] == backbone)]
    if sub.empty: continue
    best_loss = sub.loc[sub[COL].idxmax()]
    print(f'  {backbone:15s}: {LOSS_LABELS.get(best_loss["loss_type"]):<8s} Spearman={best_loss[COL]:.4f}')

print(f'\nВсе графики сохранены в: {OUTPUT_DIR}/plots/')